Data preporation LightGCN

In [23]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, log_loss, mean_squared_error
from torch.utils.data import DataLoader
from tqdm import tqdm
from itertools import combinations
from sklearn.model_selection import train_test_split

import heapq
from random import randrange
from random import seed as set_seed
import numpy as np
from numba import njit, prange
from pandas.api.types import is_numeric_dtype

import numpy as np
import pandas as pd
import torch.utils.data

from scipy.sparse.linalg import svds

In [24]:
@njit
def split_top_continuous(tasks, priorities):
    """
    Sample a sequence of unique tasks of the highest priority ensuring that
    no task will have another instance with the priority level above the lowest priority in the sequence.
    Usecases: avoiding issues with "recommendations from future" when splitting test data by timestamp.
    """
    priority_queue = [(-max(priorities), len(priorities))] # initialize typed
    priority_queue.pop()

    for idx, priority in enumerate(priorities):
        heapq.heappush(priority_queue, (-priority, idx))

    topseq = {}  # continuous sequence of top-priority tasks
    nonseq_idx = []  # top-priority tasks that interrupt continuous sequence

    unique_tasks = set(tasks)
    while unique_tasks:
        _, idx = heapq.heappop(priority_queue)
        task = tasks[idx]
        try:
            visited = topseq[task]
        except:
            unique_tasks.remove(task)
        else:
            nonseq_idx.append(visited)
        topseq[task] = idx

    topseq_idx = [idx for _, idx in topseq.items()]
    lowseq_idx = [idx for _, idx in priority_queue]  # all remaining tasks
    return topseq_idx, lowseq_idx, nonseq_idx



def to_numeric_array(series):
    if not is_numeric_dtype(series):
        if not hasattr(series, 'cat'):
            series = series.astype('category')
        return series.cat.codes.values
    return series.values


# see polara

In [25]:
def earliest_last_out(data, userid='user_id', priority='timestamp', copy=False):
    '''
    It helps avoiding "recommendations from future", when training set contains
    events that occur later than some events in the holdout and can therefore
    provide an oracle hint for the algorithm. 
    '''
    holdout_idx, observed_idx, future_idx = split_top_continuous(
        to_numeric_array(data[userid]), data[priority].values
    )
    
    observed = data.iloc[observed_idx]
    holdout = data.iloc[holdout_idx]
    future = data.iloc[future_idx]

    if copy:
        observed = observed.copy()
        holdout = holdout.copy()
        future = future.copy()
    
    return observed, holdout, future

#see polara

In [26]:
class MovieLens20MDataset(torch.utils.data.Dataset):
    """
    MovieLens 20M Dataset

    Data preparation
        treat samples with a rating less than 3 as negative samples

    :param dataset_path: MovieLens dataset path

    Reference:
        https://grouplens.org/datasets/movielens
    """

    def __init__(self, dataset_path, sep=',', engine='c', header='infer'):
        self.data = pd.read_csv(dataset_path, sep=sep, engine=engine, header=header).to_numpy()[:, :4]
        self.items = self.data[:, :2].astype(np.int32) - 1  # -1 because ID begins from 1
        self.targets = self.__preprocess_target(self.data[:, 2]).astype(np.float32)
        self.field_dims = np.max(self.items, axis=0) + 1
        self.user_field_idx = np.array((0, ), dtype=np.int64)
        self.item_field_idx = np.array((1,), dtype=np.int64)

    def __len__(self):
        return self.targets.shape[0]

    def __getitem__(self, index):
        return self.items[index], self.targets[index]

    def __preprocess_target(self, target):
        target[target <= 0] = 0
        target[target > 0] = 1
        return target


class MovieLens1MDataset(MovieLens20MDataset):
    """
    MovieLens 1M Dataset

    Data preparation
        treat samples with a rating less than 3 as negative samples

    :param dataset_path: MovieLens dataset path

    Reference:
        https://grouplens.org/datasets/movielens
    """

    def __init__(self, dataset_path):
        super().__init__(dataset_path, sep='::', engine='python', header=None)

In [27]:
dataset_path = "ml-1m/ratings.dat"
dataset = MovieLens1MDataset(dataset_path)

In [28]:
user_num = dataset.field_dims[0]
item_num = dataset.field_dims[1]
print("Number of users: ", user_num, ", Number of items: ", item_num)

Number of users:  6040 , Number of items:  3952


In [29]:
columns_name=['user_id','item_id','rating', 'timestamp']
df = pd.DataFrame(dataset.data, columns = columns_name)
df

,user_id,item_id,rating,timestamp
0,1,1193,1,978300760
1,1,661,1,978302109
2,1,914,1,978301968
3,1,3408,1,978300275
4,1,2355,1,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,1,956704887
1000206,6040,562,1,956704746
1000207,6040,1096,1,956715648


In [30]:
df_sorted = df.sort_values(by='timestamp')

# Рассчитываем размер тестовой выборки (3%)
stab_size = int(len(df_sorted) * 0.9)
stab_size1 = int(len(df_sorted) * 0.008)


# Разделяем данные на train и test
stab_train = df_sorted.head(stab_size)
new_train_1 = stab_train.tail(stab_size1)
all_train = df_sorted.tail(len(df_sorted) - stab_size)
observed, holdout, future = earliest_last_out(all_train)

train = stab_train
train_new = pd.concat([new_train_1, observed], ignore_index=True)
train_old = train.head(stab_size - stab_size1)
test = holdout
test = test.sort_values(by='timestamp')
split_index = int(len(test) * 0.25)
valid = test.iloc[:split_index]
test = test.iloc[split_index:]
test_inds = test['user_id'].unique()
valid_inds = valid['user_id'].unique()
new_users = train_new["user_id"].unique()
valid_new = set(new_users) & set(valid_inds)
test_new = set(new_users) & set(test_inds)

In [31]:
print("Dataset statistics:")
print("Train length: ", len(train_old))
print("Train users: ", len(train_old["user_id"].unique()))
print("New Train length: ", len(train_new))
print("New Train users: ", len(train_new["user_id"].unique()))
print("Val length: ", len(valid))
print("Val users: ", len(valid["user_id"].unique()))
print("Test length: ", len(test))
print("Test users: ", len(test["user_id"].unique()))
print("Valid-new intersection: ", len(valid_new))
print("Test-new intersection: ", len(test_new))

Dataset statistics:
Train length:  892187
Train users:  5956
New Train length:  8212
New Train users:  181
Val length:  302
Val users:  302
Test length:  907
Test users:  907
Valid-new intersection:  74
Test-new intersection:  63


In [32]:
train.drop(columns=['rating', 'timestamp'], inplace = True)
train_old.drop(columns=['rating', 'timestamp'], inplace = True)
train_new.drop(columns=['rating', 'timestamp'], inplace = True)
test.drop(columns=['rating','timestamp'], inplace = True)
train = np.array(train)
train_old = np.array(train_old)
train_new = np.array(train_new)
test = np.array(test)
valid = np.array(valid)
train = train.astype(np.int32) - 1
train_old = train_old.astype(np.int32) - 1
train_new = train_new.astype(np.int32) - 1
test = test.astype(np.int32) - 1
test_inds = test_inds.astype(np.int32) - 1
valid = valid.astype(np.int32) - 1
valid_inds = valid_inds.astype(np.int32) - 1


<ipython-input-32-43ca483c3e51>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train.drop(columns=['rating', 'timestamp'], inplace = True)
<ipython-input-32-43ca483c3e51>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_old.drop(columns=['rating', 'timestamp'], inplace = True)


In [33]:
train_df = pd.DataFrame(train, columns = ['user_id', 'item_id'])
train_old_df = pd.DataFrame(train_old, columns = ['user_id', 'item_id'])
train_new_df = pd.DataFrame(train_new, columns = ['user_id', 'item_id'])
test_df = pd.DataFrame(test, columns = ['user_id', 'item_id'])

In [34]:
def matrix_filler(data, matrix):
    for i in tqdm(range(data.shape[0])):
        matrix[data[i][0]][data[i][1]] = 1.0
    return matrix

In [35]:
train_mas = np.zeros((user_num, item_num))
test_mas = np.zeros((user_num, item_num))
valid_mas = np.zeros((user_num, item_num))

train_matrix = torch.tensor(matrix_filler(train, train_mas))
train_old_matrix = torch.tensor(matrix_filler(train_old, train_mas))
train_new_matrix = torch.tensor(matrix_filler(train_new, train_mas))
del train_mas
test_matrix = torch.tensor(matrix_filler(test, test_mas))
del test_mas
valid_matrix = torch.tensor(matrix_filler(valid, valid_mas))
del valid_mas

100%|██████████| 302/302 [00:00<00:00, 237829.48it/s]


Metrics

In [36]:
#Metrics

def metrics(targets, predictions, train):
    """
    Compute metrics:
    MRR
    MRR@10
    HR@1
    HR@3
    HR@10
    Parameters:
    target - real interactions
    predictions - recomendations
    """
    inf = 1e9
    predictions = predictions - train * predictions * inf
    #print(targets.shape)
    _, idx = torch.sort(predictions, dim=1, descending=True)

    targets_sorted = targets.gather(1, idx)
    ranks = (targets_sorted > 0.1).nonzero(as_tuple=False)[:, 1]
    ranks = ranks + 1
    #print((1/ranks).shape)
    mrr = torch.mean(1 / ranks)
    
    idx_10 = idx[:, :10]
    targets_sorted_10 = targets.gather(1, idx_10)
    ranks_10 = (targets_sorted_10 > 0.1).nonzero(as_tuple=False)[:, 1]
    ranks_10 = ranks_10 + 1
    mrr_10 = torch.sum(1 / ranks_10) / ranks.shape[0]
    
    cov = len(np.unique(idx_10)) / item_num

    hits = []
    for k in [1, 3, 10]:
        hits += [(ranks <= k).float().mean()]

    return {
        "mrr" : mrr,
        "mrr@10": mrr_10,
        "hits@1": hits[0],
        "hits@3": hits[1],
        "hits@10": hits[2],
        "cov": cov
    }

LightGCN

In [37]:
import pandas as pd
from sklearn import preprocessing as pp
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
import numpy as np
import random
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import time
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [38]:
def convert_to_sparse_tensor(dok_mtrx):
    
    dok_mtrx_coo = dok_mtrx.tocoo().astype(np.float32)
    values = dok_mtrx_coo.data
    indices = np.vstack((dok_mtrx_coo.row, dok_mtrx_coo.col))

    i = torch.LongTensor(indices)
    v = torch.FloatTensor(values)
    shape = dok_mtrx_coo.shape

    dok_mtrx_sparse_tensor = torch.sparse.FloatTensor(i, v, torch.Size(shape))

    return dok_mtrx_sparse_tensor

In [39]:
class LightGCN(nn.Module):
    def __init__(self, data, n_users, n_items, n_layers, latent_dim):
        super(LightGCN, self).__init__()
        self.data = data
        self.n_users = n_users
        self.n_items = n_items
        self.n_layers = n_layers
        self.latent_dim = latent_dim
        self.init_embedding()
        self.norm_adj_mat_sparse_tensor = self.get_A_tilda()

    def init_embedding(self):
        """
        Weight initialization
        """
        self.E0 = nn.Embedding(self.n_users + self.n_items, self.latent_dim)
        nn.init.xavier_uniform_(self.E0.weight)
        self.En = nn.Embedding(1, self.latent_dim)
        nn.init.xavier_uniform_(self.En.weight)
        

    def get_A_tilda(self, full_data = None):
        """
        Create Adjency matrix with normalization
        """
        R = sp.dok_matrix((self.n_users, self.n_items), dtype = np.float32)
        if full_data is None:
            R[self.data['user_id'], self.data['item_id']] = 1.0
        else:
            R[full_data['user_id'], full_data['item_id']] = 1.0

        adj_mat = sp.dok_matrix(
                (self.n_users + self.n_items, self.n_users + self.n_items), dtype=np.float32
            )
        adj_mat = adj_mat.tolil()
        R = R.tolil()

        adj_mat[: self.n_users, self.n_users :] = R
        adj_mat[self.n_users :, : self.n_users] = R.T
        adj_mat = adj_mat.todok()

        rowsum = np.array(adj_mat.sum(1))
        d_inv = np.power(rowsum + 1e-9, -0.5).flatten()
        d_inv[np.isinf(d_inv)] = 0.0
        d_mat_inv = sp.diags(d_inv)
        norm_adj_mat = d_mat_inv.dot(adj_mat)
        norm_adj_mat = norm_adj_mat.dot(d_mat_inv)
        
        # Below Code is toconvert the dok_matrix to sparse tensor.
        
        norm_adj_mat_coo = norm_adj_mat.tocoo().astype(np.float32)
        values = norm_adj_mat_coo.data
        indices = np.vstack((norm_adj_mat_coo.row, norm_adj_mat_coo.col))

        i = torch.LongTensor(indices)
        v = torch.FloatTensor(values)
        shape = norm_adj_mat_coo.shape

        norm_adj_mat_sparse_tensor = torch.sparse.FloatTensor(i, v, torch.Size(shape))

        return norm_adj_mat_sparse_tensor
    
    def propagate_through_layers(self, ind = None):
        """
        Multiply with deegrees of Laplasian. 
        If nessesary updates only one vector with index ind 
        """
        if ind is None:
            all_layer_embedding = [self.E0.weight]
            E_lyr = self.E0.weight
        else:
            E_lyr = torch.cat((self.E0.weight[:ind], self.En.weight, self.E0.weight[ind + 1:]), 0)
            all_layer_embedding = [E_lyr]
        E_lyr0 = E_lyr.clone()

        for layer in range(self.n_layers):
            E_lyr = torch.sparse.mm(self.norm_adj_mat_sparse_tensor.to(device), E_lyr.to(device))
            all_layer_embedding.append(E_lyr)

        all_layer_embedding = torch.stack(all_layer_embedding)
        mean_layer_embedding = torch.mean(all_layer_embedding, axis = 0)

        final_user_Embed, final_item_Embed = torch.split(mean_layer_embedding, [self.n_users, self.n_items])
        initial_user_Embed, initial_item_Embed = torch.split(E_lyr0, [self.n_users, self.n_items])

        return final_user_Embed, final_item_Embed, initial_user_Embed, initial_item_Embed

    def forward(self, users, pos_items, neg_items, ind = None):
        """
        Takes batch of embeddings
        """
        final_user_Embed, final_item_Embed, initial_user_Embed, initial_item_Embed = self.propagate_through_layers(ind)

        users_emb, pos_emb, neg_emb = final_user_Embed[users], final_item_Embed[pos_items], final_item_Embed[neg_items]
        userEmb0,  posEmb0, negEmb0 = initial_user_Embed[users], initial_item_Embed[pos_items], initial_item_Embed[neg_items]

        return users_emb, pos_emb, neg_emb, userEmb0,  posEmb0, negEmb0


Loss

In [40]:
def bpr_loss(users, users_emb, pos_emb, neg_emb, userEmb0,  posEmb0, negEmb0):
    """
    BPRLoss
    """
    reg_loss = (1/2)*(userEmb0.norm().pow(2) + 
                    posEmb0.norm().pow(2)  +
                    negEmb0.norm().pow(2))/float(len(users))
    pos_scores = torch.mul(users_emb, pos_emb)
    pos_scores = torch.sum(pos_scores, dim=1)
    neg_scores = torch.mul(users_emb, neg_emb)
    neg_scores = torch.sum(neg_scores, dim=1)
        
    loss = -torch.mean(torch.log(torch.nn.functional.sigmoid(-neg_scores + pos_scores)))
        
    return loss, reg_loss

Dataloader

In [41]:
class Dataloader:
    def __init__(self, data, batch_size, n_usr, n_itm):
        self.interected_items_df = data.groupby('user_id')['item_id'].apply(list).reset_index()
        self.n_usr = n_usr
        self.n_itm = n_itm
        self.batch_size = batch_size
        self.data = data
        self.indices = list(self.data['user_id'].unique())
        
    def sample_neg(self, x):
        """
        Negative sampling
        """
        while True:
            neg_id = random.randint(0, self.n_itm - 1)
            if neg_id not in x:
                return neg_id
            
    def data_loader(self):
        """
        Load batch
        """
        if len(self.indices) < self.batch_size:
            users = [random.choice(self.indices) for _ in range(self.batch_size)]
        else:
            users = random.sample(self.indices, self.batch_size)

        users.sort()
  
        users_df = pd.DataFrame(users, columns = ['users'])

        items_df = pd.merge(self.interected_items_df, users_df, how = 'right', left_on = 'user_id', right_on = 'users')

        pos_items = items_df['item_id'].apply(lambda x : random.choice(list(x))).values

        neg_items = items_df['item_id'].apply(lambda x: self.sample_neg(list(x))).values

        return list(users), list(pos_items), list(neg_items)
        
    def one_user(self, i):
        """
        Load data of only user i
        """
        users_df = pd.DataFrame([self.indices[i]], columns = ['users'])

        items_df = pd.merge(self.interected_items_df, users_df, how = 'right', left_on = 'user_id', right_on = 'users')

        pos_items = items_df['item_id'].apply(lambda x : random.choice(list(x))).values

        neg_items = items_df['item_id'].apply(lambda x: self.sample_neg(list(x))).values

        return list(users), list(pos_items), list(neg_items)
    
        

Train Full model

In [42]:
# Initialization
latent_dim = 32
n_layers = 3
lightGCN = LightGCN(train_df, user_num, item_num, n_layers, latent_dim)

optimizer = torch.optim.AdamW(lightGCN.parameters(), lr = 0.005)
EPOCHS = 5
BATCH_SIZE = 4096
DECAY = 0.0000001


In [43]:
dataload = Dataloader(train_df, BATCH_SIZE, user_num, item_num)

In [45]:
# Training procedure
loss_list_epoch = []
MF_loss_list_epoch = []
reg_loss_list_epoch = []

recall_list = []
precision_list = []
ndcg_list = []
map_list = []
hr_list = []
mrr_list = []

train_time_list = []
eval_time_list = [] 
for epoch in range(EPOCHS):
    print(epoch)
    n_batch = int(len(train)/(BATCH_SIZE))
  
    final_loss_list = []
    MF_loss_list = []
    reg_loss_list = []
  
    best_ndcg = -1
  
    train_start_time = time.time()
    lightGCN = lightGCN.to(device)
    lightGCN.train()
    for batch_idx in tqdm(range(n_batch)):

        optimizer.zero_grad()

        users, pos_items, neg_items = dataload.data_loader()


        users_emb, pos_emb, neg_emb, userEmb0,  posEmb0, negEmb0 = lightGCN.forward(users, pos_items, neg_items)


        mf_loss, reg_loss = bpr_loss(users, users_emb, pos_emb, neg_emb, userEmb0,  posEmb0, negEmb0)
        # mf_loss, reg_loss = mse_loss(users, users_emb, pos_emb, neg_emb, userEmb0,  posEmb0, negEmb0)
        reg_loss = DECAY * reg_loss
        final_loss = mf_loss + reg_loss

        final_loss.backward()
        optimizer.step()

        final_loss_list.append(final_loss.item())
        MF_loss_list.append(mf_loss.item())
        reg_loss_list.append(reg_loss.item())


    train_end_time = time.time()
    train_time = train_end_time - train_start_time

    lightGCN.eval()
    with torch.no_grad():
    
        final_user_Embed, final_item_Embed, initial_user_Embed, initial_item_Embed = lightGCN.propagate_through_layers()
        final_user_Embed = final_user_Embed.cpu()
        final_item_Embed = final_item_Embed.cpu()
        initial_user_Embed = initial_user_Embed.cpu()
        initial_item_Embed = initial_item_Embed.cpu()
        
        scores = torch.matmul(final_user_Embed / torch.linalg.norm(final_user_Embed), torch.transpose(final_item_Embed,0, 1) / torch.linalg.norm(final_item_Embed))
        metr_test = metrics(test_matrix[test_inds], scores[test_inds], train_matrix[test_inds])
        hr_list += [metr_test['hits@10'].item()]
        print("Test:")
        print(metr_test)
        metr_valid = metrics(valid_matrix[valid_inds], scores[valid_inds], train_matrix[valid_inds])
        print("Valid:")
        print(metr_valid)

    eval_time = time.time() - train_end_time

    loss_list_epoch.append(round(np.mean(final_loss_list),4))
    MF_loss_list_epoch.append(round(np.mean(MF_loss_list),4))
    reg_loss_list_epoch.append(round(np.mean(reg_loss_list),4))


    train_time_list.append(train_time)
    eval_time_list.append(eval_time)
    
    


0


  0%|          | 0/219 [00:00<?, ?it/s]

Test:
{'mrr': tensor(0.0350), 'mrr@10': tensor(0.0237), 'hits@1': tensor(0.0121), 'hits@3': tensor(0.0254), 'hits@10': tensor(0.0606), 'cov': 0.17130566801619435}
Valid:
{'mrr': tensor(0.0184), 'mrr@10': tensor(0.0084), 'hits@1': tensor(0.), 'hits@3': tensor(0.0132), 'hits@10': tensor(0.0298), 'cov': 0.13081983805668015}
1


  0%|          | 0/219 [00:00<?, ?it/s]

Test:
{'mrr': tensor(0.0339), 'mrr@10': tensor(0.0224), 'hits@1': tensor(0.0121), 'hits@3': tensor(0.0243), 'hits@10': tensor(0.0573), 'cov': 0.19331983805668015}
Valid:
{'mrr': tensor(0.0206), 'mrr@10': tensor(0.0107), 'hits@1': tensor(0.0033), 'hits@3': tensor(0.0166), 'hits@10': tensor(0.0265), 'cov': 0.14903846153846154}
2


...

Test:
{'mrr': tensor(0.0349), 'mrr@10': tensor(0.0235), 'hits@1': tensor(0.0121), 'hits@3': tensor(0.0254), 'hits@10': tensor(0.0617), 'cov': 0.2074898785425101}
Valid:
{'mrr': tensor(0.0212), 'mrr@10': tensor(0.0117), 'hits@1': tensor(0.0033), 'hits@3': tensor(0.0199), 'hits@10': tensor(0.0364), 'cov': 0.159665991902834}
3


  0%|          | 0/219 [00:00<?, ?it/s]

Test:
{'mrr': tensor(0.0358), 'mrr@10': tensor(0.0248), 'hits@1': tensor(0.0121), 'hits@3': tensor(0.0221), 'hits@10': tensor(0.0706), 'cov': 0.2479757085020243}
Valid:
{'mrr': tensor(0.0210), 'mrr@10': tensor(0.0109), 'hits@1': tensor(0.0033), 'hits@3': tensor(0.0066), 'hits@10': tensor(0.0397), 'cov': 0.18496963562753035}
4


  0%|          | 0/219 [00:00<?, ?it/s]

Test:
{'mrr': tensor(0.0341), 'mrr@10': tensor(0.0240), 'hits@1': tensor(0.0099), 'hits@3': tensor(0.0221), 'hits@10': tensor(0.0728), 'cov': 0.2785931174089069}
Valid:
{'mrr': tensor(0.0188), 'mrr@10': tensor(0.0088), 'hits@1': tensor(0.), 'hits@3': tensor(0.0132), 'hits@10': tensor(0.0397), 'cov': 0.19838056680161945}


Training on only old interactions

In [50]:
# Initialization
latent_dim = 32
n_layers = 3
lightGCN = LightGCN(train_old_df, user_num, item_num, n_layers, latent_dim)

optimizer = torch.optim.AdamW(lightGCN.parameters(), lr = 0.005)
EPOCHS = 5
BATCH_SIZE = 4096
DECAY = 0.0000001

In [51]:
dataload = Dataloader(train_old_df, BATCH_SIZE, user_num, item_num)

In [52]:
# Main Train
loss_list_epoch = []
MF_loss_list_epoch = []
reg_loss_list_epoch = []

recall_list = []
precision_list = []
ndcg_list = []
map_list = []
hr_list = []
mrr_list = []

train_time_list = []
eval_time_list = [] 
best_hr = -1
for epoch in range(EPOCHS):
    print(epoch)
    n_batch = int(len(train_old)/(BATCH_SIZE))
  
    final_loss_list = []
    MF_loss_list = []
    reg_loss_list = []
  
    best_ndcg = -1
  
    train_start_time = time.time()
    lightGCN = lightGCN.to(device)
    lightGCN.train()
    for batch_idx in tqdm(range(n_batch)):

        optimizer.zero_grad()

        users, pos_items, neg_items = dataload.data_loader()

        users_emb, pos_emb, neg_emb, userEmb0,  posEmb0, negEmb0 = lightGCN.forward(users, pos_items, neg_items)
        
        mf_loss, reg_loss = bpr_loss(users, users_emb, pos_emb, neg_emb, userEmb0,  posEmb0, negEmb0)
        reg_loss = DECAY * reg_loss
        final_loss = mf_loss + reg_loss #+ 0.001 * mf_loss_mse

        final_loss.backward()
        optimizer.step()

        final_loss_list.append(final_loss.item())
        MF_loss_list.append(mf_loss.item())
        reg_loss_list.append(reg_loss.item())


    train_end_time = time.time()
    train_time = train_end_time - train_start_time

    print(lightGCN.E0.weight.norm())

    lightGCN.eval()
    with torch.no_grad():
    
        final_user_Embed, final_item_Embed, initial_user_Embed, initial_item_Embed = lightGCN.propagate_through_layers()
        final_user_Embed = final_user_Embed.cpu()
        final_item_Embed = final_item_Embed.cpu()
        initial_user_Embed = initial_user_Embed.cpu()
        initial_item_Embed = initial_item_Embed.cpu()
        
        scores = torch.matmul(final_user_Embed, torch.transpose(final_item_Embed,0, 1))
        metr_test = metrics(test_matrix[test_inds], scores[test_inds], train_matrix[test_inds])
        print("Test:")
        print(metr_test)
        if metr_test['hits@10'].item() > best_hr:
            E = lightGCN.E0.weight.data.clone()
            best_hr = metr_test['hits@10'].item()
            print("Upd")
        metr_valid = metrics(valid_matrix[valid_inds], scores[valid_inds], train_matrix[valid_inds])
        print("Valid:")
        print(metr_valid)

    eval_time = time.time() - train_end_time

    loss_list_epoch.append(round(np.mean(final_loss_list),4))
    MF_loss_list_epoch.append(round(np.mean(MF_loss_list),4))
    reg_loss_list_epoch.append(round(np.mean(reg_loss_list),4))

    train_time_list.append(train_time)
    eval_time_list.append(eval_time)


0


  0%|          | 0/217 [00:00<?, ?it/s]

tensor(198.3141, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Test:
{'mrr': tensor(0.0385), 'mrr@10': tensor(0.0273), 'hits@1': tensor(0.0176), 'hits@3': tensor(0.0287), 'hits@10': tensor(0.0628), 'cov': 0.13360323886639677}
Upd
Valid:
{'mrr': tensor(0.0188), 'mrr@10': tensor(0.0099), 'hits@1': tensor(0.0033), 'hits@3': tensor(0.0132), 'hits@10': tensor(0.0298), 'cov': 0.10602226720647773}
1


  0%|          | 0/217 [00:00<?, ?it/s]

tensor(232.5821, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Test:
{'mrr': tensor(0.0345), 'mrr@10': tensor(0.0239), 'hits@1': tensor(0.0121), 'hits@3': tensor(0.0254), 'hits@10': tensor(0.0639), 'cov': 0.16852226720647773}
Upd
Valid:
{'mrr': tensor(0.0177), 'mrr@10': tensor(0.0085), 'hits@1': tensor(0.0033), 'hits@3': tensor(0.0099), 'hits@10': tensor(0.0232), 'cov': 0.12854251012145748}
2


...

tensor(267.6694, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Test:
{'mrr': tensor(0.0362), 'mrr@10': tensor(0.0250), 'hits@1': tensor(0.0154), 'hits@3': tensor(0.0254), 'hits@10': tensor(0.0595), 'cov': 0.19686234817813766}
Valid:
{'mrr': tensor(0.0171), 'mrr@10': tensor(0.0078), 'hits@1': tensor(0.0033), 'hits@3': tensor(0.0099), 'hits@10': tensor(0.0199), 'cov': 0.15435222672064777}
3


  0%|          | 0/217 [00:00<?, ?it/s]

tensor(311.0053, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Test:
{'mrr': tensor(0.0361), 'mrr@10': tensor(0.0247), 'hits@1': tensor(0.0154), 'hits@3': tensor(0.0243), 'hits@10': tensor(0.0617), 'cov': 0.2209008097165992}
Valid:
{'mrr': tensor(0.0193), 'mrr@10': tensor(0.0099), 'hits@1': tensor(0.0033), 'hits@3': tensor(0.0132), 'hits@10': tensor(0.0265), 'cov': 0.1720647773279352}
4


  0%|          | 0/217 [00:00<?, ?it/s]

tensor(358.0095, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Test:
{'mrr': tensor(0.0340), 'mrr@10': tensor(0.0235), 'hits@1': tensor(0.0088), 'hits@3': tensor(0.0232), 'hits@10': tensor(0.0761), 'cov': 0.25025303643724695}
Upd
Valid:
{'mrr': tensor(0.0190), 'mrr@10': tensor(0.0088), 'hits@1': tensor(0.0033), 'hits@3': tensor(0.0066), 'hits@10': tensor(0.0265), 'cov': 0.19003036437246965}


Safe and Loadweights

In [ ]:
torch.save(E, "LightGCN_weights2.pt")

In [ ]:
E = torch.load("LightGCN_weights2.pt")

Check

In [53]:

lightGCN2 = LightGCN(train_old_df, user_num, item_num, n_layers, latent_dim)
lightGCN2.E0.weight = nn.Parameter(E)
print(torch.linalg.norm(E - lightGCN2.E0.weight))

lightGCN2.eval()
with torch.no_grad():
        final_user_Embed, final_item_Embed, initial_user_Embed, initial_item_Embed = lightGCN2.propagate_through_layers()
        final_user_Embed = final_user_Embed.cpu()
        final_item_Embed = final_item_Embed.cpu()
        initial_user_Embed = initial_user_Embed.cpu()
        initial_item_Embed = initial_item_Embed.cpu()
        
        scores = torch.matmul(final_user_Embed, torch.transpose(final_item_Embed,0, 1))
        metr_test = metrics(test_matrix[test_inds], scores[test_inds], train_matrix[test_inds])
        print(metr_test)

tensor(0., device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
{'mrr': tensor(0.0340), 'mrr@10': tensor(0.0235), 'hits@1': tensor(0.0088), 'hits@3': tensor(0.0232), 'hits@10': tensor(0.0761), 'cov': 0.25025303643724695}


Folding In with SGD

In [57]:
# Initialization
latent_dim = 32
n_layers = 3
lightGCN = LightGCN(train_df, user_num, item_num, n_layers, latent_dim)
lightGCN.E0.weight = nn.Parameter(E)
lightGCN.to(device)
optimizer = torch.optim.AdamW(lightGCN.parameters(), lr = 0.005)
EPOCHS = 3
BATCH_SIZE = 10
DECAY = 0.0000001
DECAY2 = 5 # Similarity to mean embedding

In [58]:
dataload = Dataloader(train_new_df, 10, user_num, item_num)

For Each User train its embedding only with AdamW and sampled embeddings 

In [61]:

mas = []
ind = 0
lightGCN = lightGCN.to(device)
vec_change = []
lightGCN.E0.weight = lightGCN.E0.weight.requires_grad_(False)
for us in tqdm(new_users):
    us = int(us)
    nn.init.xavier_uniform_(lightGCN.En.weight)
    #lightGCN.En.weight.to(device)
    a = torch.linalg.norm(lightGCN.En.weight).item()
    lightGCN.E0.weight.requires_grad_(False)
    for epoch in range(EPOCHS):
        optimizer.zero_grad()

        users, pos_items, neg_items = dataload.data_loader()


        users_emb, pos_emb, neg_emb, userEmb0,  posEmb0, negEmb0 = \
             lightGCN.forward(users, pos_items, neg_items, us)


        mf_loss, reg_loss = bpr_loss(users, users_emb, pos_emb, neg_emb, userEmb0,  posEmb0, negEmb0)

        reg_loss = DECAY * reg_loss
        final_loss = mf_loss + reg_loss + DECAY2 * torch.linalg.norm(lightGCN.En.weight - torch.mean(lightGCN.E0.weight))


        final_loss.backward()
        optimizer.step()
    b = torch.linalg.norm(lightGCN.En.weight).item()
    ind += 1
    if torch.isnan(torch.tensor(abs(b - a))):
        mas += [lightGCN.E0.weight[us].clone()]
        vec_change += [0]
    else:
        mas += [lightGCN.En.weight.clone()]
        vec_change += [abs(b - a)]
        

print(np.mean(vec_change))
        
print(torch.linalg.norm(E - lightGCN.E0.weight))       

with torch.no_grad():
    for ind, us in tqdm(enumerate(new_users)):
        lightGCN.E0.weight[us] = mas[ind]

lightGCN.eval()
with torch.no_grad():
    
        final_user_Embed, final_item_Embed, initial_user_Embed, initial_item_Embed = lightGCN.propagate_through_layers()
        final_user_Embed = final_user_Embed.cpu()
        final_item_Embed = final_item_Embed.cpu()
        initial_user_Embed = initial_user_Embed.cpu()
        initial_item_Embed = initial_item_Embed.cpu()
        
        scores = torch.matmul(final_user_Embed, torch.transpose(final_item_Embed,0, 1))
        metr_test = metrics(test_matrix[test_inds], scores[test_inds], train_matrix[test_inds])
        print("Test:")
        print(metr_test)
        metr_valid = metrics(valid_matrix[valid_inds], scores[valid_inds], train_matrix[valid_inds])
        print("Valid:")
        print(metr_valid)

  0%|          | 0/181 [00:00<?, ?it/s]

0.016785784979551535
tensor(0., device='cuda:0')


0it [00:00, ?it/s]

Test:
{'mrr': tensor(0.0348), 'mrr@10': tensor(0.0247), 'hits@1': tensor(0.0099), 'hits@3': tensor(0.0254), 'hits@10': tensor(0.0783), 'cov': 0.25101214574898784}
Valid:
{'mrr': tensor(0.0197), 'mrr@10': tensor(0.0099), 'hits@1': tensor(0.0033), 'hits@3': tensor(0.0066), 'hits@10': tensor(0.0364), 'cov': 0.18294534412955465}
